In [1]:
import sympy as sp
from sympy.physics.mechanics import ReferenceFrame, dynamicsymbols, msubs
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

# Problem Set 5

[PSet 5](MIT8_01F16_pset5.pdf)

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return ReferenceFrame(
        frame,
        latexs=(
            rf"\; {{}}^\mathcal {frame} \hat {x}",
            rf"\;{{}}^\mathcal {frame} \hat {y}",
            rf"\: {{}}^\mathcal {frame} \hat {z}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

# Problem Set 5

[Problem Set 5](./MIT8_01F16_pset5.pdf)

## Problem 5.1 Stopping a bullet

![Stopping a bullet](../figures/PS0501-Stopping-a-Bullet.jpg)


A bullet of mass $m_1$ traveling horizontally with speed $u$ hits a block of 
mass $m_2$ that is originally at rest and becomes embedded in the block. 
After the collision, the block slides horizontally a distance d on a surface with 
friction, and then falls off the surface at a height $h$ as shown.

The coefficient of kinetic friction between the block and the surface is $\mu_k$. 
Assume the collision is nearly instantaneous and all distances are 
large compared to the size of the block. Neglect air resistance.

 (a) What is $u_\text{min}$, the minimum speed of the bullet so that the 
 block falls off the surface? Express your answer in terms of some or all 
 of the following: $m_1, m_2, \mu_k, d, h$ and $g$ for the gravitational constant.

 (b) Assume that the initial speed of the bullet u is large enough for the 
 block to fall off the surface. Calculate $x_f$, the position where the block 
 hits the ground measured from the bottom edge of the surface. 
 
 Express your answer in terms of some or all
of the following: $m_1$, $m_2$, $\mu_k$, $u$, $d$, $h$ and $g$

In [3]:
m1, m2, muk, u, d, h, g = sp.symbols("m_1 m_2 \mu_k u d h g ", real=True, positive=True)

N = reference_frame("N")

# Initial state of the system
vec_u = u * N.x
pb = m1 * vec_u

# Final state of the system
ua = sp.symbols("u_a", real=True, positive=True)
vec_uf = ua * N.x
pf = m1 * vec_uf + m2 * vec_uf

display(Math(r"\text{Initial momentum: } \mathbf{p}_b =  " + sp.latex(pb)))
display(Math(r"\text{Final momentum: } \mathbf{p}_f =  " + sp.latex(pf)))

# Ignore friction force of collision, so momentum is conserved
eq_momentum = sp.Eq(pb.to_matrix(N), pf.to_matrix(N))
display(Math(r"\text{Momentum conservation: } " + sp.latex(eq_momentum)))

ua_solution = sp.solve(eq_momentum, ua, dict=True)[0]

display(
    Math(r"\text{Final velocity of the system: } u_a = " + sp.latex(ua_solution[ua]))
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
# N2L
N_BS = sp.symbols("N_{BS}", real=True, positive=True)
vec_N_BS = N_BS*(-N.y)
f_BS = sp.symbols("f_{BS}", real=True, positive=True)
vec_f_BS = f_BS*(-N.x)

vec_F_total = (m1 + m2)*g*N.y  + vec_N_BS + vec_f_BS
display(Math(r"\text{Total force on the system: } \mathbf{F}_{total} = " + sp.latex(vec_F_total)))

acc_b = sp.symbols("acc_b", real=True)
vec_acc_b = acc_b*N.x
display(Math(r"\text{Acceleration of the system: } \mathbf{a}_b = " + sp.latex(vec_acc_b)))
eq_acc = sp.Eq(vec_F_total.to_matrix(N), (m1 + m2)*vec_acc_b.to_matrix(N))
display(Math(r"\text{Newton's 2nd law: } " + sp.latex(eq_acc)))

N2Lsol = sp.solve(eq_acc, [acc_b, N_BS], dict=True)[0]
acceleration = msubs(N2Lsol[acc_b], {f_BS: muk*N2Lsol[N_BS]}).simplify()
display(Math(r"\text{Acceleration of the system: } a_b = " + sp.latex(acceleration)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
# a.
kinematic_sol = sp.solve(sp.Eq(0**2 - ua_solution[ua]**2, 2*acceleration*d), u)
kinematic_sol = [sol for sol in kinematic_sol if sol.is_real and sol > 0][0]
display(Math(r"\text{Initial velocity of the system: } u = " + sp.latex(kinematic_sol)))

<IPython.core.display.Math object>

In [6]:
#b.
vi, a, s = sp.symbols("v_i a s", real=True)
t_h = sp.symbols("t_h", real=True, positive=True)

vf = sp.sqrt(vi**2 + 2*a*s)
vf = msubs(vf, {vi: ua_solution[ua], a: acceleration, s: d})
display(Math(r"\text{Velocity at edge: } v_f = " + sp.latex(vf)))

# Time to drop by h
eqn_h  = sp.Eq(s, (g*t_h**2)/2)
eqn_h = msubs(eqn_h, {s: h})
display(Math(r"\text{Equation for time to drop by height: } " + sp.latex(eqn_h)))
sol_t_h = sp.solve(eqn_h, t_h)
sol_t_h = [sol for sol in sol_t_h if sol.is_real and sol > 0][0]
display(Math(r"\text{Time to drop by height: } t_h = " + sp.latex(sol_t_h)))

xf = (sol_t_h * vf)
display(Math(r"\text{Distance traveled horizontally: } x_f = " + sp.latex(xf)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>